### Data Profiling and Quality Checks

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Data Profiling and Quality Checks") \
    .getOrCreate()
print(spark.sparkContext._jsc.sc().listJars())

Vector()


Extract Data Sources

In [2]:
movies_df = spark.read.csv(r"project_data/Movies_main.csv", header=True, inferSchema=True)
ratings_df = spark.read.json(r"project_data/ratings.json")
extended_df = spark.read.csv(r"project_data/movie_extended.csv", header=True, inferSchema=True)


### Data Profiling in Movies Data Frame

In [3]:
movies_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: double (nullable = true)



In [4]:
from pyspark.sql.functions import col, sum
print(f"Total Rows: {movies_df.count()}")
movies_df.describe().show()

Total Rows: 45486
+-------+------------------+--------------------+------------+--------------------+--------------------+
|summary|                id|               title|release_date|              budget|             revenue|
+-------+------------------+--------------------+------------+--------------------+--------------------+
|  count|             45486|               45480|       45396|               41051|               45480|
|   mean|108374.20594507839|            Infinity|        NULL|   3787077.809837024| 1.120586919001759E7|
| stddev|112478.20956852814|                 NaN|        NULL|1.6498644873845305E7|6.4318739131462455E7|
|    min|               100|!Women Art Revolu...|  01-01-1890|/ff9qCepilowshEtG...|                 0.0|
|    max|              9999|    ファンタスティポ|  31-12-2015|              998000|       2.787965087E9|
+-------+------------------+--------------------+------------+--------------------+--------------------+



### Uniqueness Check

In [5]:
duplicates = movies_df.groupBy("id") \
            .count() \
            .filter(col("count") > 1)
print(f"Duplicate Rows: {duplicates.count()}")

Duplicate Rows: 49


### Completeness Check

In [6]:
from pyspark.sql.functions import col, sum, length, min as spark_min, max as spark_max

# Total number of rows
total_rows = movies_df.count()

# Compute missing value count per column
missing_values_df = movies_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in movies_df.columns])
missing_values_df.show()

+---+-----+------------+------+-------+
| id|title|release_date|budget|revenue|
+---+-----+------------+------+-------+
|  0|    6|          90|  4435|      6|
+---+-----+------------+------+-------+



## ID - Accuracy Check


In [7]:
id_accuracy_df = movies_df.select("id").withColumn("count_string", length(col("id")).cast("string"))

#checker_df.show()
id_min_max = id_accuracy_df.agg(
    spark_min("count_string").alias("min_length"),
    spark_max("count_string").alias("max_length")
)
id_min_max.show()

# Find non-integer IDs (IDs that contain letters or special characters)
invalid_ids = movies_df.filter(~col("id").rlike("^[0-9]+$"))

invalid_ids.show(125)

+----------+----------+
|min_length|max_length|
+----------+----------+
|         1|         6|
+----------+----------+

+----------+-----+------------+--------------------+-------+
|        id|title|release_date|              budget|revenue|
+----------+-----+------------+--------------------+-------+
|1997-08-20| NULL|        NULL|/ff9qCepilowshEtG...|   NULL|
|2012-09-29| NULL|        NULL|                   0|   NULL|
|2014-01-01| NULL|        NULL|/zaSf5OG7V8X8gqFv...|   NULL|
+----------+-----+------------+--------------------+-------+



### Title Accuracy Check

In [8]:

# Titles should have at least one letter (allowing numbers, but not only symbols)
invalid_titles = movies_df.filter(~col("title").rlike(".*[a-zA-Z].*"))

invalid_titles.show(125)

+------+--------------------+------------+---------+------------+
|    id|               title|release_date|   budget|     revenue|
+------+--------------------+------------+---------+------------+
| 54285|            301, 302|  22-04-1995|     NULL|         0.0|
|101230|               1-900|  01-09-1994|        0|         0.0|
|   422|                  8½|  1963-02-14|        0|         0.0|
|  3682|                  54|  1998-08-28| 13000000| 1.6757163E7|
|  4437|                2010|  1984-12-06| 28000000| 4.0400657E7|
| 47817|                1969|  18-08-1988|        0|   5979011.0|
| 14902|                1776|  11/09/1972|        0|         0.0|
|  9051|                  10|  1979-10-04|        0| 7.4865517E7|
| 11519|                1941|  1979-12-13| 35000000| 3.1755742E7|
|  3870|                1900|  03-09-1976|  9000000|         0.0|
|   844|                2046|  2004-05-20| 12000000| 1.9271312E7|
| 20536|                 61*|  28-04-2001|        0|         0.0|
|  9282|  

### Date Accuracy Check

In [9]:
from pyspark.sql.functions import col

# Detect different date formats
formats_df = movies_df.select(
    col("release_date"),
    col("release_date").rlike("^\d{1,2}/\d{1,2}/\d{4}$").alias("MM/DD/YYYY"),
    col("release_date").rlike("^\d{2}-\d{2}-\d{4}$").alias("DD-MM-YYYY"),
    col("release_date").rlike("^\d{4}-\d{2}-\d{2}$").alias("YYYY-MM-DD")
)

from pyspark.sql.functions import col, sum

# Aggregate counts for each format
count_df = formats_df.agg(
    sum(col("MM/DD/YYYY").cast("int")).alias("MM/DD/YYYY Count"),
    sum(col("DD-MM-YYYY").cast("int")).alias("DD-MM-YYYY Count"),
    sum(col("YYYY-MM-DD").cast("int")).alias("YYYY-MM-DD Count")
)

count_df.show()

+----------------+----------------+----------------+
|MM/DD/YYYY Count|DD-MM-YYYY Count|YYYY-MM-DD Count|
+----------------+----------------+----------------+
|           15067|           15135|           15194|
+----------------+----------------+----------------+



#### Budget - Accuracy

In [10]:
from pyspark.sql.functions import col

# Find movies with missing or zero budget
zero_budget_df = movies_df.filter((col("budget") == 0))
print(f"Count of 0 Budget: {zero_budget_df.count()}")
non_integer_budget = movies_df.filter(~col("budget").rlike("^[0-9]+$")).count()
print(f"Count of Non-Integer Budget: {non_integer_budget}")
max_budget = movies_df.agg(spark_max("budget"),spark_min("budget"))
print(f"Max Budget: {max_budget}")

Count of 0 Budget: 33887
Count of Non-Integer Budget: 2
Max Budget: DataFrame[max(budget): string, min(budget): string]


In [11]:
from pyspark.sql.functions import col, max as spark_max, min as spark_min

# Convert 'budget' column to integer
movies_df = movies_df.withColumn("budget", col("budget").cast("double"))

# Find movies with missing or zero budget
zero_budget_df = movies_df.filter(col("budget") == 0)
print(f"Count of 0 Budget: {zero_budget_df.count()}")

# Find non-numeric budgets (originally stored as strings)
non_numeric_budget = movies_df.filter(col("budget").isNull()).count()
print(f"Count of Non-Numeric Budget: {non_numeric_budget}")

# Get max and min budget values correctly
max_min_budget = movies_df.agg(
    spark_max(col("budget")).alias("max_budget"),
    spark_min(col("budget")).alias("min_budget")
).collect()[0]  # Extract values from DataFrame row

print(f"Max Budget: {max_min_budget['max_budget']}")
print(f"Min Budget: {max_min_budget['min_budget']}")


Count of 0 Budget: 33887
Count of Non-Numeric Budget: 4437
Max Budget: 380000000.0
Min Budget: 0.0


### Revenue - Accuracy

In [12]:
from pyspark.sql.functions import col

# Find movies with missing or zero budget
movies_df =movies_df.withColumn("revenue", col("revenue").cast("double"))

zero_revenue_df = movies_df.filter(col("revenue") == 0)
print(f"Count of 0 Revenue: {zero_revenue_df.count()}")

non_numeric_revenue = movies_df.filter(col("revenue").isNull()).count()
print(f"Count of Non-Numeric Revenue: {non_numeric_revenue}")

max_min_revenue = movies_df.agg(
    spark_max(col("revenue")).alias("max_revenue"),
    spark_min(col("revenue")).alias("min_revenue")
).collect()[0]  

print(f"Max Revenue: {max_min_revenue['max_revenue']}")
print(f"Min Revenue: {max_min_revenue['min_revenue']}")

Count of 0 Revenue: 38068
Count of Non-Numeric Revenue: 6
Max Revenue: 2787965087.0
Min Revenue: 0.0


### Movie_extended.csv

#### ID

In [13]:
extended_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)



In [14]:
extended_df.describe().show()

+-------+------------------+--------------------+--------------------+--------------------+--------------------+
|summary|                id|              genres|production_companies|production_countries|    spoken_languages|
+-------+------------------+--------------------+--------------------+--------------------+--------------------+
|  count|             45466|               43024|               33101|               45466|               45460|
|   mean|108359.91881310077|                NULL|              1201.0|   5.766666666666667|                NULL|
| stddev| 112460.7492781323|                NULL|                NULL|  1.3650396819628847|                NULL|
|    min|               100|              Action|      (주)로드픽쳐스|                 4.3|                  []|
|    max|              9999|Western,War,Histo...|           프로덕션M|[{'iso_3166_1': '...|[{'iso_639_1': 'z...|
+-------+------------------+--------------------+--------------------+--------------------+---------------

In [33]:
id_extended_df = extended_df.select("id").withColumn("count_string", length(col("id")).cast("string"))
id_extended_duplicates = extended_df.groupBy("id") \
            .count() \
            .filter(col("count") > 1)
#id_extended_duplicates.show()
print(f"Duplicate Rows: {id_extended_duplicates.count()}")

# Accuracy/Validity/Integrity Checks
id_extended_acc_min_max = id_extended_df.select(length(col("id")).alias("length")) \
                    .agg(spark_min("length").alias("min_char_length"),spark_max("length").alias("max_char_length"))
id_extended_acc_min_max.show() 

id_extended_df.select(length(col("id")).alias("length"), "id").filter(col("length") == 10).show()

Duplicate Rows: 29
+---------------+---------------+
|min_char_length|max_char_length|
+---------------+---------------+
|              1|             10|
+---------------+---------------+

+------+----------+
|length|        id|
+------+----------+
|    10|1997-08-20|
|    10|2012-09-29|
|    10|2014-01-01|
+------+----------+



In [34]:
id_extended_acc_format = movies_df.filter(~col("id").rlike("^[0-9]+$"))
id_extended_acc_format.show()
spark.stop()

+----------+-----+------------+------+-------+
|        id|title|release_date|budget|revenue|
+----------+-----+------------+------+-------+
|1997-08-20| NULL|        NULL|  NULL|   NULL|
|2012-09-29| NULL|        NULL|   0.0|   NULL|
|2014-01-01| NULL|        NULL|  NULL|   NULL|
+----------+-----+------------+------+-------+

